# 15 — CSTR OCP and receding-horizon NMPC

**Objectives.** Build a scaled multiple-shooting OCP, apply the first input, shift the horizon, and warm-start the next solve. NMPC repeats: measure/estimate state → update parameter → solve → apply one control → shift. Bounds are safety/model-validity constraints, not tuning weights.


In [ ]:
import casadi as ca
import numpy as np
from casadi_tutorial.models.cstr import CSTRParameters,cstr_rk4_function
p=CSTRParameters(); N=10; dt=0.05; F=cstr_rk4_function(dt,p)
opti=ca.Opti(); X=opti.variable(2,N+1); U=opti.variable(1,N); x0=opti.parameter(2)
opti.subject_to(X[:,0]==x0)
for k in range(N): opti.subject_to(X[:,k+1]==F(X[:,k],U[:,k],p.nominal_disturbance))
opti.subject_to(opti.bounded(0,X[0,:],12)); opti.subject_to(opti.bounded(285,X[1,:],360))
opti.subject_to(opti.bounded(285,U,315))
dx=ca.diag(ca.DM([1/0.5,1/5]))@(X-ca.repmat(ca.DM(p.nominal_state),1,N+1))
du=(U-p.nominal_jacket_temperature)/10
opti.minimize(ca.sumsqr(dx)+0.1*ca.sumsqr(du))
opti.solver('ipopt',{'print_time':False},{'print_level':0,'max_iter':300,'tol':1e-7})


In [ ]:
state=p.nominal_state+np.array([0.25,-1.0]); closed_loop=[state.copy()]; controls=[]
for iteration in range(5):
    opti.set_value(x0,state)
    if iteration==0: opti.set_initial(X,np.tile(state[:,None],(1,N+1))); opti.set_initial(U,p.nominal_input)
    solution=opti.solve(); last_success=solution.stats()['success']; predicted_x=solution.value(X); predicted_u=solution.value(U)
    control=float(np.ravel(predicted_u)[0]); controls.append(control)
    state=np.asarray(F(state,[control],p.nominal_disturbance).full()).ravel(); closed_loop.append(state.copy())
    opti.set_initial(X,np.column_stack([predicted_x[:,1:],predicted_x[:,-1]]))
    opti.set_initial(U,np.column_stack([np.atleast_2d(predicted_u)[:,1:],np.atleast_2d(predicted_u)[:,-1]]))
print('states:',np.asarray(closed_loop),'controls:',controls)
assert last_success and np.isfinite(closed_loop).all()


Warm starts reuse primal trajectories; lower-level interfaces can also reuse `lam_x0`/`lam_g0` when solver options support it. If the measured state violates a hard path bound, the NLP is infeasible at node zero—use a deliberate soft constraint, not a mysterious tolerance increase.

**Debug checklist:** evaluate dynamics at the initial guess, plot each residual block, scale states/inputs, inspect `return_status`, and compare predicted versus applied dynamics. **Recap:** NMPC reliability comes from formulation, initialization, and validation around the solver call.
